[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansensio/axr/blob/master/axr/00_intro.ipynb)

In [ ]:
import numpy as np
import random

In [ ]:
class EntornoDados:

    META       = 20  
    PASO_RIVAL = 3  
    ESPERAR  = 0   
    AVANZAR  = 1  

    def reset(self):
        self.pos_agente = 0   # Posición inicial del agente
        self.pos_rival  = 0   # Posición inicial del rival
        return self._estado() # El estado inicial es 0 (empate)

    def _estado(self):
        return self.pos_agente - self.pos_rival

    def step(self, accion):
        
      
        if accion == self.ESPERAR:
            self.pos_agente += 2
        else:
            dado = random.randint(1, 6)
            self.pos_agente += dado

        self.pos_rival += self.PASO_RIVAL

        agente_llego = self.pos_agente >= self.META
        rival_llego  = self.pos_rival  >= self.META

        if agente_llego and rival_llego:
            return self._estado(), 0.0, True

        if agente_llego:
            return self._estado(), +1.0, True

        if rival_llego:
            return self._estado(), -1.0, True

        return self._estado(), 0.0, False

In [ ]:
class AgenteGradiente:
    NUM_ESTADOS  = 39
    NUM_ACCIONES = 2    
    OFFSET       = 19  

    def __init__(self, alpha=0.1, prob_exp =True):
        self.alpha          = alpha
        self.prob_exp  = prob_exp

        self.H = np.zeros((self.NUM_ESTADOS, self.NUM_ACCIONES))

        self.recompensa_promedio = 0.0
        self.total_episodios     = 0

    def _idx(self, estado):
        return estado + self.OFFSET

    def _softmax(self, estado):
        idx  = self._idx(estado)
        x    = self.H[idx]          
        x    = x - np.max(x)        
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x)  

    def seleccionar_accion(self, estado):
        
        prob   = self._softmax(estado)
        accion = np.random.choice(self.NUM_ACCIONES, p=prob)  
        return accion, prob

    def actualizar(self, estado, accion, prob, recompensa):
        
        idx = self._idx(estado)

    
        recompensa_lineal = self.recompensa_promedio if self.usar_recompensa else 0.0

    
        delta = recompensa - recompensa_lineal

        for a in range(self.NUM_ACCIONES):
            if a == accion:
                self.H[idx, a] += self.alpha * delta * (1 - prob[a])
            else:
                self.H[idx, a] -= self.alpha * delta * prob[a]

        self.total_episodios     += 1
        self.recompensa_promedio += (recompensa - self.recompensa_promedio) / self.total_episodios

    def politica(self, estado):
        prob = self._softmax(estado)
        return np.argmax(prob) 

El estado representa lo que el agente percibe del entorno en cada momento
Aqu el estado es la diferencia de posicion entre el agente y el rival:
estado = posicion_agente - posicion_rival

Estado negativo : el agente va detras del rival
Estado 0 : están empatados
Estado positivo : el agente va adelante

El rango de estados posibles es [-19, 19] (39 estados distintos)

Entorno
El entorno es el tablero de 20 casillas con el rival determinista
Responde a las acciones del agente actualizando las posiciones y calculando la recompensa

El agente tiene exactamente 2 acciones posibles en cada turno
 0 : ESPERAR: avanza 2 casillas (sin riesgo)
 1 : AVANZAR: lanza el dado (avance aleatorio de 1–6 casillas)

Politica
La politica define qué acción tomar en cada estado.  
Usando el Algoritmo del Gradiente, la política es estocastica: asigna una *probabilidad* a cada acción usando la función Softmax sobre las preferencias aprendidas H(estado, acción).
Recompensa
La recompensa se otorga al final de cada episodio (partida):
- +1 si el agente gana (llega a la casilla 20 antes que el rival).
- -1 si el agente pierde.
- 0 si hay empate(ambos llegan en el mismo turno).


Reglas del juego
- El tablero tiene 20 casillas. El primero en llegar gana.
- En cada turno el agente elige una de dos acciones:
  - AVANZAR: lanza un dado (1–6). Avanza esa cantidad de casillas.
  - ESPERAR: no lanza. Avanza exactamente 2 casillas.
- El rival siempre avanza 3 casillas fijas por turno.
- El agente aprende con el tiempo qué acción conviene más según su posición relativa.